# Split the benchmark by question type — synthesis vs lookup

Reads the `benchmark_report.json` you already produced, asks gpt-oss to label each
question as **synthesis** (needs combining two or more documents/sections/facts) or
**lookup** (a single fact in one place), then recomputes all four metrics — accuracy,
time, input tokens, output tokens — as mean and standard deviation for both systems,
split by bucket and overall.

The arithmetic is done here in pandas (exact, over every row), not by the model — the
LLM only classifies one question at a time, with caching, so it never has to hold the
whole set at once. That is why the earlier all-in-one prompt stopped after a few
questions; this does not have that failure mode.

The point: find whether TreeRAG's accuracy gap narrows on the synthesis bucket, the
subset where a hierarchy should help and dense top-k retrieval is weak.

Outputs `benchmark_by_type.json` (per-bucket stats) and `questions_by_type.csv`
(every question with its label and per-system metrics).

In [5]:
%pip install ollama numpy pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os, json, re, time, ast
from pathlib import Path
import numpy as np, pandas as pd

OLLAMA_URL  = "http://localhost:11528"
JUDGE_MODEL = "gpt-oss:120b"
REPORT_FILE = "benchmark_report.json"     # the report from prototype_benchmark
OUT_JSON    = "benchmark_by_type.json"
OUT_CSV     = "questions_by_type.csv"
KEEP_ALIVE  = "30m"
print(f"classifier {JUDGE_MODEL}; reading {REPORT_FILE}")

classifier gpt-oss:120b; reading benchmark_report.json


In [7]:
import ollama
client = ollama.Client(host=OLLAMA_URL, timeout=600)

def _names(r):
    raw=r.get("models",[]) if hasattr(r,"get") else getattr(r,"models",[])
    out=[]
    for m in raw:
        n=getattr(m,"model",None) or getattr(m,"name",None)
        if n is None and isinstance(m,dict): n=m.get("model") or m.get("name")
        if n: out.append(n)
    return out

def _wait():
    said=False
    while True:
        try:
            if any(JUDGE_MODEL in n for n in _names(client.list())):
                print(f"ollama ok; {JUDGE_MODEL} loaded"); return
            why=f"{JUDGE_MODEL} not loaded yet"
        except Exception as e: why=f"unreachable; {type(e).__name__}: {e}"
        if not said: print(f"waiting for ollama, {why}; rechecking every 10s"); said=True
        time.sleep(10)

def ask(prompt, num_predict=200, think=False):
    attempt=0; pass_think=True
    while True:
        kw=dict(model=JUDGE_MODEL, messages=[{"role":"user","content":prompt}],
                options={"temperature":0,"num_predict":num_predict}, keep_alive=KEEP_ALIVE)
        if pass_think: kw["think"]=think
        try:
            r=client.chat(**kw)
            t=(r["message"]["content"] or "").strip()
            if not t:
                try: t=(r["message"]["thinking"] or "").strip()
                except Exception: pass
            return t
        except TypeError: pass_think=False
        except Exception as e:
            attempt+=1
            if attempt==1 or attempt%5==0: print(f"[waiting for ollama] {type(e).__name__}: {e}; retrying")
            time.sleep(min(60,5*2**min(attempt-1,4)))

def _bar(total, done, desc):
    try:
        from tqdm.auto import tqdm; return tqdm(total=total, initial=done, desc=desc)
    except Exception:
        class _S:
            def __init__(s): s.n=done; print(f"{desc}: {done}/{total}")
            def update(s,n=1):
                s.n+=n
                if s.n==total or s.n%20==0: print(f"  {desc}: {s.n}/{total}")
            def close(s): print(f"{desc}: done ({total}/{total})")
            def set_postfix(s,**k): pass
        return _S()

_wait()

ollama ok; gpt-oss:120b loaded


In [8]:
# load the per-question rows from the benchmark report
raw=Path(REPORT_FILE).read_text(encoding="utf-8")
try: report=json.loads(raw)
except Exception: report=ast.literal_eval(raw)
per_q=pd.DataFrame(report["per_question"])
need=["id","question","tree_accuracy","qms_accuracy","tree_time","qms_time",
      "tree_in","qms_in","tree_out","qms_out"]
missing=[c for c in need if c not in per_q.columns]
assert not missing, f"report is missing columns: {missing}"
print(f"loaded {len(per_q)} questions from {REPORT_FILE}")

loaded 324 questions from benchmark_report.json


In [9]:
# classify one question as synthesis or lookup; cached so reruns and tunnel drops are cheap
CLS_FILE=Path("split_cache/labels.json"); LABELS={}
if CLS_FILE.exists(): LABELS=json.loads(CLS_FILE.read_text())
def _csave():
    CLS_FILE.parent.mkdir(exist_ok=True)
    tmp=CLS_FILE.with_suffix(".tmp"); tmp.write_text(json.dumps(LABELS)); tmp.replace(CLS_FILE)

def classify(question):
    prompt=("classify this question into exactly one bucket.\n"
            "- synthesis: answering needs combining or cross-referencing information from two or more distinct "
            "documents, sections, or facts; comparison, multi-hop, multi-part conditions, or reconciling sources.\n"
            "- lookup: answering needs a single fact, value, definition, location, or procedure from one place.\n"
            "when genuinely ambiguous, choose lookup.\n\n"
            f"QUESTION: {question}\n\n"
            'reply with ONLY json: {"type": "synthesis" or "lookup", "why": "<one clause>"}')
    t=ask(prompt,num_predict=120)
    m=re.search(r'\{.*\}', t, re.S)
    if m:
        try:
            o=json.loads(m.group(0)); lab=str(o.get("type","")).lower()
            if "synth" in lab: return "synthesis", str(o.get("why",""))[:160]
            return "lookup", str(o.get("why",""))[:160]
        except Exception: pass
    return ("synthesis" if "synth" in t.lower() else "lookup"), "fallback parse"

ids=list(per_q["id"])
bar=_bar(len(ids), sum(1 for i in ids if str(i) in LABELS), "classifying")
n_s=sum(1 for v in LABELS.values() if v["type"]=="synthesis")
for _,r in per_q.iterrows():
    qid=str(r["id"])
    if qid not in LABELS:
        typ,why=classify(r["question"]); LABELS[qid]={"type":typ,"why":why}; _csave(); bar.update(1)
    s=sum(1 for v in LABELS.values() if v["type"]=="synthesis")
    bar.set_postfix(synthesis=s, lookup=len(LABELS)-s)
bar.close()

per_q["type"]=per_q["id"].astype(str).map(lambda i: LABELS[i]["type"])
per_q["type_why"]=per_q["id"].astype(str).map(lambda i: LABELS[i]["why"])
print("\nlabel counts:")
print(per_q["type"].value_counts().to_string())

/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
classifying: 100%|███████████████████████████████████████| 324/324 [22:07<00:00,  4.10s/it, lookup=213, synthesis=111]


label counts:
type
lookup       213
synthesis    111


In [10]:
# eyeball the labels; the whole conclusion rests on these being right
print("a few of each label:\n")
for t in ["synthesis","lookup"]:
    print(f"--- {t} ---")
    for _,r in per_q[per_q["type"]==t].head(6).iterrows():
        print(f"  [{r['id']}] {r['question'][:80]}")
        print(f"       -> {r['type_why']}")
    print()

a few of each label:

--- synthesis ---
  [QMS-9] How does Continuous Improvement work at OICR?
       -> fallback parse
  [QMS-10] Which departments are responsible for understanding and following QMS procedures
       -> fallback parse
  [QMS-11] When should you file an Incident Report?
       -> fallback parse
  [QMS-16] When is a competency assessment needed?
       -> fallback parse
  [QMS-18] What is the formal process for changing an existing SOP?
       -> fallback parse
  [QMS-21] How are KPI tracked:
       -> fallback parse

--- lookup ---
  [QMS-1] Where do you find SOPs?
       -> The question asks for a single piece of information—the location where SOPs can be found.
  [QMS-2] What version of an SOP should be used?
       -> fallback parse
  [QMS-3] How do you initiate the CAPA process?
       -> fallback parse
  [QMS-4] What conditions count as a non-conformance?
       -> fallback parse
  [QMS-5] What is the main purpose of Proficiency Testing (PT)?
       -> The quest

In [11]:
# exact metric stats per bucket, computed in pandas over every row
METRICS={"accuracy":("tree_accuracy","qms_accuracy"),
         "time_sec":("tree_time","qms_time"),
         "input_tokens":("tree_in","qms_in"),
         "output_tokens":("tree_out","qms_out")}
def _ms(s):
    s=pd.to_numeric(s,errors="coerce").dropna()
    return round(float(s.mean()),3), round(float(s.std(ddof=1)) if len(s)>1 else 0.0,3)

def bucket_stats(df):
    out={}
    for name,(tc,qc) in METRICS.items():
        tm,ts=_ms(df[tc]); qm,qs=_ms(df[qc])
        out[name]={"treerag_mean":tm,"treerag_std":ts,"qms_mean":qm,"qms_std":qs,
                   "mean_diff_tree_minus_qms":round(tm-qm,3)}
    return out

buckets={"overall":per_q,
         "synthesis":per_q[per_q["type"]=="synthesis"],
         "lookup":per_q[per_q["type"]=="lookup"]}
stats={b:{"n":int(len(df)),"metrics":bucket_stats(df)} for b,df in buckets.items() if len(df)}

for b,info in stats.items():
    print(f"\n=== {b} (n = {info['n']}) ===")
    rows=[{"metric":m,**info["metrics"][m]} for m in METRICS]
    print(pd.DataFrame(rows)[["metric","treerag_mean","treerag_std","qms_mean","qms_std","mean_diff_tree_minus_qms"]].to_string(index=False))


=== overall (n = 324) ===
       metric  treerag_mean  treerag_std  qms_mean  qms_std  mean_diff_tree_minus_qms
     accuracy         0.293        0.384     0.546    0.436                    -0.253
     time_sec       147.417      211.391    82.540  160.055                    64.877
 input_tokens      7584.302     1690.389   474.253    6.490                  7110.049
output_tokens      5011.386     1251.992    61.062   11.852                  4950.324

=== synthesis (n = 111) ===
       metric  treerag_mean  treerag_std  qms_mean  qms_std  mean_diff_tree_minus_qms
     accuracy         0.308        0.368     0.543    0.423                    -0.235
     time_sec       134.363       50.134    74.130   81.296                    60.233
 input_tokens      7771.153     1716.425   476.216    8.048                  7294.937
output_tokens      5289.730     1311.831    63.649   13.909                  5226.081

=== lookup (n = 213) ===
       metric  treerag_mean  treerag_std  qms_mean  qms_st

In [12]:
# the headline: where is treerag closest to or ahead of qms on accuracy?
def acc(b): return stats[b]["metrics"]["accuracy"]
print("accuracy gap (treerag - qms), by bucket:")
for b in ["overall","synthesis","lookup"]:
    if b in stats:
        a=acc(b); gap=a["mean_diff_tree_minus_qms"]
        flag="  <-- treerag closest/ahead here" if b!="overall" and gap==max(acc(x)["mean_diff_tree_minus_qms"] for x in stats if x!="overall") else ""
        print(f"  {b:10s} n={stats[b]['n']:>4}   treerag {a['treerag_mean']:.3f}   qms {a['qms_mean']:.3f}   gap {gap:+.3f}{flag}")
if "synthesis" in stats and "lookup" in stats:
    gs=acc("synthesis")["mean_diff_tree_minus_qms"]; gl=acc("lookup")["mean_diff_tree_minus_qms"]
    print(f"\nsynthesis gap {gs:+.3f} vs lookup gap {gl:+.3f}: "
          + ("treerag does relatively BETTER on synthesis, the foothold you were looking for."
             if gs>gl else "treerag does not do relatively better on synthesis here."))

accuracy gap (treerag - qms), by bucket:
  overall    n= 324   treerag 0.293   qms 0.546   gap -0.253
  synthesis  n= 111   treerag 0.308   qms 0.543   gap -0.235  <-- treerag closest/ahead here
  lookup     n= 213   treerag 0.285   qms 0.547   gap -0.262

synthesis gap -0.235 vs lookup gap -0.262: treerag does relatively BETTER on synthesis, the foothold you were looking for.


In [13]:
# write outputs
Path(OUT_JSON).write_text(json.dumps({"by_bucket":stats,
    "labels":{str(r['id']):{"type":r['type'],"why":r['type_why']} for _,r in per_q.iterrows()}}, indent=2))
keep=["id","type","question","tree_accuracy","qms_accuracy","accuracy_delta",
      "tree_time","qms_time","tree_in","qms_in","tree_out","qms_out","type_why"]
per_q[[c for c in keep if c in per_q.columns]].to_csv(OUT_CSV,index=False)
print(f"wrote {OUT_JSON} and {OUT_CSV}")

wrote benchmark_by_type.json and questions_by_type.csv
